<style>
/* 마크다운 내 일반 텍스트(p)와 리스트(li)의 줄간격을 1.8로 강제 적용 */
.markdown-rendered p, 
.markdown-rendered li {
    line-height: 2.0 !important;
    font-size: 3.0em !important;
}
</style>


# 📘 반도체 패키징 역설계 핵심 기법 & 알고리즘 상세 해설서

이 문서는 AI 기반 반도체 패키징 역설계 파이프라인(Step 1 ~ Step 5)에 사용된 데이터 사이언스 및 전산역학(CAE) 기법들을 입문자의 눈높이에 맞추어 상세한 비유와 함께 정리한 백서입니다.

---

## [Step 1] 대리 모델 학습 및 데이터 증강

### 1. 절댓값 Max Peak 추출 (`abs().idxmax()`)
* **1. 기법 소개:** 시간에 따라 위아래로 요동치는 파형 데이터에서, 방향(플러스, 마이너스)에 상관없이 가장 파괴적인 충격이 발생한 순간을 찾아내는 전처리 기법입니다. 단순히 양수 중에 가장 큰 것을 찾는 것이 아니라 물리적인 힘의 절대적인 크기 자체를 평가합니다. 이를 통해 엔지니어는 시뮬레이션 중 가장 가혹한 조건이 언제 발생했는지 정확히 특정할 수 있습니다.
* **2. 주요 특징:** 단순한 최댓값(max)이나 최솟값(min) 함수를 단독으로 사용할 때 발생하는 정보 누락을 완벽하게 방지합니다. 데이터에 절댓값을 씌워 크기를 비교하여 위치를 찾고 나면, 다시 원본 데이터가 가진 부호(+, -)를 살려서 가져오는 것이 핵심입니다. 이렇게 하면 해당 응력이 팽창으로 인한 것인지(+) 수축으로 인한 것인지(-)에 대한 중요한 물리적 방향성을 잃지 않습니다.
* **3. 작동 원리:** 예를 들어 `[10, -30, 20]`이라는 지진파 혹은 응력 데이터가 들어왔다고 가정해 보겠습니다. 알고리즘은 먼저 모든 값에 절댓값을 씌워 `[10, 30, 20]`으로 만든 뒤, 가장 큰 숫자인 30의 위치를 찾습니다. 그다음 원본 배열로 돌아가 해당 위치에 있는 진짜 값인 `-30`을 최종 결과물로 뽑아내는 2단계 방식을 거쳐 파괴력을 보존합니다.
* **4. 프로젝트 도입 이유:** 패키징이 가열될 때는 기판이 위로 휘며 인장 응력(+)이 발생하지만, 냉각될 때는 아래로 심하게 휘며 치명적인 압축 응력(-)이 발생합니다. 단순 최댓값(max) 연산을 사용하면 이 위험한 압축 응력을 시스템이 무시해버리는 대참사가 발생하므로 이를 막기 위해 사용했습니다.
* **5. 📺 참고 영상:** [유튜브: 파이썬 데이터 전처리 기초 (결측치, 이상치, 파생변수)](https://www.youtube.com/results?search_query=파이썬+데이터+전처리+기초+강의)

### 2. 피어슨 상관계수 (Pearson Correlation Coefficient)
* **1. 기법 소개:** 두 개의 데이터 흐름이 얼마나 찰떡같이 서로를 따라다니는지, 즉 선형적인 관계가 있는지를 -1에서 1 사이의 숫자로 명확하게 보여주는 통계 기법입니다. 데이터 분석을 시작할 때 변수들 간의 관계를 파악하는 가장 기초적이면서도 강력한 도구로 쓰입니다. 복잡한 수식을 몰라도 두 변수의 친밀도를 직관적으로 수치화하여 확인할 수 있습니다.
* **2. 주요 특징:** 결과값이 1에 가까울수록 두 데이터가 같은 방향으로 강하게 움직인다는 뜻(정비례)입니다. 반대로 -1에 가까울수록 하나가 커질 때 다른 하나는 작아지는 정반대의 움직임(반비례)을 보입니다. 0에 가까우면 한 변수가 어떻게 변하든 다른 변수에는 아무런 영향을 주지 않는 완전한 남남이라는 의미를 갖습니다.
* **3. 작동 원리:** "부모의 키가 크면 자녀의 키도 클까?"라는 질문을 수학적으로 푸는 원리와 동일합니다. 두 변수가 각각 자신의 평균값으로부터 얼마나 떨어져 있는지(분산)를 구하고, 이를 곱해서 두 변수가 함께 변하는 정도(공분산)를 측정합니다. 마지막으로 이 값들을 각자의 표준편차로 나누어 스케일을 -1과 1 사이로 예쁘게 평탄화시켜 줍니다.
* **4. 프로젝트 도입 이유:** 본격적으로 딥러닝과 같은 블랙박스 AI를 학습시키기 전에 데이터의 인과관계를 살피기 위함입니다. "덮개(P5)를 두껍게 만들면 휨(WarpMax) 현상도 같이 커진다"는 전산역학적 진실을 엔지니어가 히트맵(Heatmap)을 통해 눈으로 직접 확인하고 팩트 체크를 진행하기 위해 도입했습니다.
* **5. 📺 참고 영상:** [유튜브: 상관계수 10분 만에 완벽 이해하기](https://www.youtube.com/results?search_query=피어슨+상관계수+쉽게+설명)

### 3. GPR (가우시안 프로세스 회귀, Gaussian Process Regression) 
* **1. 기법 소개:** 데이터 점들을 하나의 딱딱한 막대기나 방정식으로 잇는 것이 아니라, 그 점들을 통과할 수 있는 **'무수히 많은 부드러운 곡선들의 확률 분포'**를 통째로 예측하는 고급 베이지안 머신러닝 알고리즘입니다. 하나의 절대적인 정답을 내놓는 대신 확률적인 가능성의 범위를 제시하는 우아한 접근 방식을 가집니다. 특히 데이터가 적을 때 그 진가를 발휘하는 강력한 보간 알고리즘입니다.
* **2. 주요 특징:** GPR의 가장 위대한 점은 예측값($\mu$) 하나만 덜렁 뱉어내고 끝나는 것이 아니라, **"내가 이 예측을 얼마나 확신하는가?"를 나타내는 오차 범위(불확실성, $\sigma$)**를 반드시 함께 알려준다는 것입니다. 학습 데이터가 촘촘한 곳에서는 확신에 차서 좁은 오차 범위를 주지만, 데이터가 없는 미지의 영역으로 갈수록 오차 범위를 넓게 잡아 스스로의 무지를 겸손하게 인정합니다.
* **3. 작동 원리:** 마치 탐험가가 지도를 그리는 과정과 비슷합니다. 이미 우리가 관측한 산봉우리(실제 Ansys 데이터)의 높이는 100% 확신하므로 오차가 0입니다. 하지만 아직 가보지 않은 미지의 골짜기는 주변 봉우리들의 높이와 거리를 참고하여 "대략 이 정도 높이일 확률이 높지만, 위아래로 10미터 정도 틀릴 수도 있다"라며 주변 데이터를 바탕으로 확률 지도를 유연하게 업데이트해 나갑니다.
* **4. 프로젝트 도입 이유:** 기존에 널리 쓰이는 트리 모델(XGBoost 등)은 훈련 데이터를 벗어난 미지의 영역(외삽)을 만나면 예측값이 뚝 끊기거나 양 끝단에 뾰족한 벽(Clipping 뿔)을 만들어버리는 치명적인 구조적 한계가 있습니다. 이를 극복하고 물리 법칙에 맞는 부드럽고 연속적인 다차원 응력 곡면을 시뮬레이션하기 위해 전산역학 대리 모델로 선정했습니다.
본 프로젝트의 경우 데이터셋의 수량이 적어 연산 시간이 매우 길다는 GPR의 단점이 상쇄된다.
* **5. 📺 참고 영상:** [유튜브: 가우시안 프로세스 회귀 직관적 이해](https://www.youtube.com/results?search_query=가우시안+프로세스+회귀+설명)

### 4. ARD 커널 (Automatic Relevance Determination)
* **1. 기법 소개:** GPR 모델이 데이터를 바라보고 학습할 때 **"어떤 설계 변수가 핵심 뼈대 역할을 하고, 어떤 변수가 쓸모없는 쓰레기 데이터인지" 스스로 깨닫게 만드는 똑똑한 수학적 돋보기(커널)**입니다. 입력된 여러 개의 변수 중에서 결과에 미치는 기여도를 모델이 알아서 가중치로 변환합니다. 차원이 늘어날수록 유용해지는 필수적인 하이퍼파라미터 튜닝 기법입니다.
* **2. 주요 특징:** 데이터를 바라보는 '거리의 척도(Length Scale)'를 6개의 차원(P1~P6)마다 독립적으로 다르게 부여합니다. 즉, 각 치수별로 예민함을 다르게 설정하는 기능입니다. 이 기능이 없으면 AI는 6개의 변수가 모두 똑같이 중요하다고 착각하는 등방성(Isotropic)의 오류에 빠져 정밀한 물리 현상을 잡아내지 못합니다.
* **3. 작동 원리:** AI가 학습을 반복하면서, 결과값에 아무런 영향을 주지 않는 둔감한 변수 쪽으로는 눈금(Length Scale)을 무한히 늘려버립니다. 눈금이 커지면 값이 아무리 변해도 AI는 "아까랑 똑같네"라고 무시해 버립니다. 반대로 결과에 치명적인 영향을 주는 예민한 변수는 눈금을 아주 짧게 좁혀서 현미경처럼 미세한 변화도 날카롭게 캐치해 냅니다.
* **4. 프로젝트 도입 이유:** 6개의 두께 변수(P1~P6) 중 휨과 박리를 지배하는 진짜 범인(예: P5 덮개 두께)을 AI가 수학적으로 정확히 분리해 내기 위함입니다. 이를 통해 모델은 불필요한 노이즈 변수에 휘둘리지 않고 진짜 중요한 축에 집중하여 예측의 정확도를 99% 이상으로 끌어올릴 수 있었습니다.
* **5. 📺 참고 영상:** [유튜브: 커널 함수와 머신러닝의 관계 완벽 정리](https://www.youtube.com/results?search_query=머신러닝+커널+함수+쉽게+이해)

### 5. 라틴 하이퍼큐브 샘플링 (LHS, Latin Hypercube Sampling)
* **1. 기법 소개:** 다차원의 복잡한 우주 공간에 탐색용 점들을 뿌릴 때, 주사위를 마구잡이로 던지는 단순 랜덤(Random) 방식의 맹점을 극복하는 기법입니다. 공간을 가장 겹치지 않고 골고루 흩뿌려 채우는 **고급 난수(랜덤) 생성 알고리즘**입니다. 적은 수의 샘플링으로도 전체 공간의 특성을 매우 훌륭하게 대변할 수 있게 해주는 통계학의 꽃입니다.
* **2. 주요 특징:** 단순 랜덤 샘플링을 하면 우연히 특정 구역에만 데이터가 뭉치고 어떤 구역은 텅 비어버리는 사각지대(Dead Zone)가 필연적으로 발생합니다. LHS는 이런 뭉침 현상을 수학적으로 원천 차단합니다. 따라서 생성된 가상 데이터들이 6차원 탐색 공간 전체를 촘촘하고 공평하게 지배하도록 강제하는 강력한 제어력을 가지고 있습니다.
* **3. 작동 원리:** 숫자가 겹치지 않게 넣는 '스도쿠 퍼즐'이나 체스의 '룰(Rook) 말 배치'를 생각하면 쉽습니다. 각 차원(가로, 세로, 높이 등)을 균등한 눈금을 가진 격자로 쪼갭니다. 그런 다음, 어떤 가로줄이나 세로줄을 훑어보더라도 반드시 데이터 점이 딱 1개씩만 들어가도록 매우 엄격하게 자리를 배치하여 골고루 섞어줍니다.
* **4. 프로젝트 도입 이유:** 대리 모델(AI)에게 예측을 시켜볼 10만 개의 거대한 가상 조합을 만들 때, 이 기법이 반드시 필요합니다. 특정 두께 조합만 편식해서 생성되는 것을 막고, P1~P6의 6차원 탐색 공간 전체 구석구석을 사각지대 없이 촘촘하게 찔러보기 위해서 사용했습니다.
* **5. 📺 참고 영상:** [유튜브: 몬테카를로 시뮬레이션과 샘플링 기법의 차이](https://www.youtube.com/results?search_query=라틴+하이퍼큐브+샘플링+설명)

---

## [Step 2] 은닉 제약조건 분류기 (Gatekeeper)

### 6. 랜덤 포레스트 분류기 (Random Forest Classifier) 

[Image of Random Forest algorithm]

* **1. 기법 소개:** 수백 개의 얕고 단순한 '스무고개 질문 트리(의사결정 나무)'를 마치 숲(Forest)처럼 무성하게 모아놓는 알고리즘입니다. 새로운 데이터가 들어오면 각 트리가 낸 의견을 다수결로 투표하여 최종 결론을 내리는 가장 대표적인 앙상블(Ensemble) 머신러닝 기법입니다. 구조가 직관적이면서도 예측력이 뛰어나 현업에서 가장 많이 쓰이는 모델 중 하나입니다.
* **2. 주요 특징:** 하나의 똑똑하고 복잡한 나무(트리)는 자신이 본 학습 데이터만 달달 외워버려 새로운 문제에 대처하지 못하는 편견(과적합)에 빠지기 매우 쉽습니다. 하지만 조금씩 다르게 훈련받은 수백 명의 평범한 나무들이 모여 다수결 투표를 진행하면, 개별 나무의 편견과 오류가 상쇄되며 훨씬 정확하고 튼튼한 '집단 지성'이 발휘됩니다.
* **3. 작동 원리:** "기판이 1mm보다 얇아?", "덮개가 2mm보다 두꺼워?"와 같은 조건문으로 스무고개를 하는 나무들을 100그루 심습니다. 새로운 설계 도면의 치수가 입력되면 100그루의 나무가 각자의 기준대로 판별한 뒤 "터진다(Fail)" 혹은 "안전하다(Safe)" 표지판을 듭니다. 시스템은 이 표지판을 세어 더 많이 표를 얻은 쪽으로 최종 판정을 내립니다.
* **4. 프로젝트 도입 이유:** 시뮬레이션의 격자(Mesh)가 열팽창을 견디지 못하고 깨지며 에러(Fail)가 나는 물리적 파괴 조합의 경계선은 매우 불규칙하고 울퉁불퉁합니다. 이런 비선형적인 지뢰밭의 패턴을 선형 모델로는 잡을 수 없기 때문에, 이 패턴을 가장 부드럽고 튼튼하게 감싸서 막아내는 방패 역할을 부여하기 위해 도입했습니다.
* **5. 📺 참고 영상:** [유튜브: 랜덤 포레스트 개념 직관적 이해 (강추)](https://www.youtube.com/results?search_query=랜덤포레스트+쉽게+이해하기+허민석)

### 7. 클래스 불균형 해소 (`class_weight='balanced'`)
* **1. 기법 소개:** 데이터의 승패 비율이 심각하게 한쪽으로 쏠려 있을 때(예: 정상 95%, 불량 5%), AI가 "복잡하게 계산할 거 없이 전부 정상이라고 찍으면 무조건 95점이네?"라고 꼼수를 부리는 현상을 원천 차단하는 학습 교정 기법입니다. 머신러닝이 소외된 소수의 데이터에도 귀를 기울이도록 강제하는 수학적 장치입니다.
* **2. 주요 특징:** 절대적인 샘플 개수가 적은 소수(Minority) 데이터 클래스 쪽에 막대한 페널티 가중치를 부여합니다. 이를 통해 머신러닝의 손실 함수(Loss Function) 공간을 뒤틀어, 기울어진 운동장을 AI의 관점에서 완전히 평평하게 만들어 줍니다.
* **3. 작동 원리:** 시험에서 1점짜리 쉬운 문제(정상 데이터) 95개와 100점짜리 초고난도 문제(불량 데이터) 5개가 있다고 가정해 보겠습니다. AI가 불량 데이터를 하나라도 틀리면, 정상 데이터를 틀렸을 때보다 수학적으로 훨씬 더 아프게 벌점(Loss)을 맞도록 내부 계산 규칙을 바꿉니다. 이렇게 하면 AI는 벌점을 피하기 위해 불량 데이터를 필사적으로 찾아내는 방향으로 진화하게 됩니다.
* **4. 프로젝트 도입 이유:** 1,200개의 원본 데이터 중 해석이 터진 위험한 데이터(Fail)는 고작 25%에 불과합니다. 하지만 이 25%의 파탄 조합은 실제 공정에서 대형 사고로 이어지는 핵심 감시 대상이므로, AI가 다수결의 함정에 빠지지 않고 파탄 조합을 하나라도 놓치지 않게 만들기 위해 사용했습니다.
* **5. 📺 참고 영상:** [유튜브: 불균형 데이터 처리 방법론과 가중치 부여](https://www.youtube.com/results?search_query=불균형+데이터+머신러닝+처리)

---

## [Step 3] 파레토 프론티어 타겟 곡선 추출

### 8. 파레토 비지배 정렬 (Pareto Non-dominated Sorting) 
* **1. 기법 소개:** "가장 가벼우면서도, 배터리가 가장 오래가는 노트북"처럼 서로 완전히 상충하는 두 마리 토끼를 다 잡아야 하는 다목적 최적화 상황에서 쓰이는 필터링 기법입니다. 데이터들끼리 무한 경쟁을 시켜 **누구에게도 절대 꿀리지 않는 '절대적 1등 설계들의 집합'을 솎아내는 수학적 서열 필터**입니다. 
* **2. 주요 특징:** 단순히 두 목적의 점수를 더해서 평균을 내는 꼼수(가중합, Weighted Sum)를 쓰지 않습니다. 휨 최소화와 박리 최소화라는 두 가지 목적 함수 각각을 전혀 타협하지 않는 독립된 잣대로 유지한 채로, 진정한 의미의 최상위 계급(Frontier)을 찾아냅니다. 이 방법은 설계자에게 다양한 1등 옵션들을 제공한다는 큰 장점이 있습니다.
* **3. 작동 원리:** A라는 설계가 B보다 '휨'도 적고 '박리 응력'도 적다면 수학적으로 "A가 B를 완벽하게 지배(Dominate)한다"고 판정하며 열등한 B를 목록에서 도태시킵니다. 이렇게 수만 개의 데이터를 1:1 데스매치 풀리그로 싸우게 한 뒤, 그 어떤 다른 데이터에게도 지배당하지 않고 살아남은 무적의 녀석들만 모아 'Frontier 0(1등 그룹)'으로 묶습니다.
* **4. 프로젝트 도입 이유:** AI에게 역설계를 시킬 때 "적당히 이런 파동대로 설계해 줘"라고 타협할 수는 없습니다. 현실 물리 세계에서 달성 가능하면서도 가성비가 가장 완벽하게 조율된 궁극의 물리적 '정답지(유토피아 타겟)'를 실제 원본 데이터들 속에서 발굴해 내기 위해 이 엄격한 정렬 기준을 사용했습니다.
* **5. 📺 참고 영상:** [유튜브: 다목적 최적화와 파레토 프론티어의 개념](https://www.youtube.com/results?search_query=파레토+프론티어+다목적+최적화+설명)

---

## [Step 4] 1D-CNN 오토인코더 잠재 매핑 역설계

### 9. 1D-CNN 오토인코더 (Autoencoder) 
* **1. 기법 소개:** 입력된 방대한 데이터를 아주 작은 병목 공간(Bottleneck)으로 욱여넣어 압축(Encoder)했다가, 이 압축된 정보만을 가지고 다시 원래의 모습으로 똑같이 복원(Decoder)하는 훈련을 반복하는 딥러닝 기술입니다. 이 가혹한 훈련을 통해 AI는 불필요한 노이즈를 버리고 **'데이터의 진짜 엑기스 요약본'을 스스로 터득하게 됩니다.**
* **2. 주요 특징:** 사람이 정답을 알려주지 않아도 자기 자신을 똑같이 베끼는 '비지도 학습(Unsupervised Learning)'을 수행합니다. 특히 우리가 사용한 1D-CNN 구조는 사진(2차원)이 아닌 시간의 흐름(1차원)에 따라 진동하는 파동 패턴을 스캔하고 압축하는 데 타의 추종을 불허하는 성능을 보여줍니다.
* **3. 작동 원리:** 4,319개(600타임스텝 x 7채널)나 되는 시계열 숫자 뭉치를 합성곱(Convolution) 렌즈로 훑어내어 단 32개의 1차원 암호(잠재 벡터, Latent Vector)로 극강의 압축을 진행합니다. 그런 다음, 반대쪽 디코더 네트워크가 이 32개의 암호만 보고 원래의 4,319개 파동 모양을 똑같이 그려내며 자신의 압축 성능을 증명합니다.
* **4. 프로젝트 도입 이유:** 수천 개의 복잡한 응력 파동 숫자를 한 번에 6개의 두께(P1~P6)로 다이렉트 역추적하면 딥러닝 모델이 인지 과부하에 걸려 수렴하지 못합니다. 그래서 파동을 32개의 '핵심 뼈대'로 예쁘게 압축한 뒤, 이 가벼운 뼈대를 보고 두께를 유추하는 안정적인 2단계 우회 매핑 작전을 수행하기 위해 도입했습니다.
* **5. 📺 참고 영상:** [유튜브: 오토인코더(Autoencoder) 10분 만에 이해하기](https://www.youtube.com/results?search_query=오토인코더+쉽게+이해하기)

### 10. 다층 퍼셉트론 (MLP) 역매핑
* **1. 기법 소개:** 다층 퍼셉트론(Multi-Layer Perceptron)은 인간의 뇌 신경망 구조를 수학적으로 모방한 가장 기초적이면서도 범용적인 딥러닝 모델입니다. 복잡하게 얽혀 있는 입력값과 출력값 사이의 숨겨진 비선형 수학 공식을 인공 신경망이 스스로 찾아내어 연결해 주는 역할을 수행합니다.
* **2. 주요 특징:** 신경세포 역할을 하는 수많은 노드(Node)와 층(Layer)을 여러 겹으로 쌓아 올립니다. 데이터가 층을 통과할 때마다 가중치를 곱하고 활성화 함수(비선형 함수)를 통과시키면서, 단순한 1차 방정식으로는 절대 풀 수 없는 극도로 복잡한 패턴을 학습할 수 있는 능력을 갖춥니다.
* **3. 작동 원리:** 입력 데이터가 주어지면 네트워크를 따라 계산을 거쳐 예측값을 내놓습니다. 이 예측값이 실제 정답과 얼마나 틀렸는지 오차를 계산한 뒤, 이 오차를 뒤로 거슬러 보내며(역전파, Backpropagation) 신경망 내부의 수많은 가중치 밸브들을 미세하게 조절합니다. 이 훈련을 수만 번 반복하면 점점 오차가 0에 가까워집니다.
* **4. 프로젝트 도입 이유:** 앞서 오토인코더가 뽑아준 32개의 고도로 압축된 파동 암호를 보고, "아! 이 암호 배열이면 기판(P1) 두께는 0.8, 덮개(P5)는 1.5여야 물리 법칙이 성립해"라고 실제 물리적 치수로 번역해 주는 최종 역설계 통역기로 사용하기 위해서입니다.
* **5. 📺 참고 영상:** [유튜브: 딥러닝 인공신경망(MLP)의 작동 원리](https://www.youtube.com/results?search_query=다층퍼셉트론+딥러닝+기초+강의)

---

## [Step 5] NSGA-II 기반 통합 강건 최적화

### 11. NSGA-II (다목적 유전 알고리즘)
* **1. 기법 소개:** 다윈의 진화론(교배, 돌연변이, 적자생존) 원리를 컴퓨터 알고리즘으로 완벽하게 모방하여, 수백 세대에 걸쳐 가장 뛰어난 변수 조합을 찾아내는 생물학적 메타 휴리스틱 최적화 알고리즘입니다. 글로벌 스케일의 복잡한 최적화 문제를 푸는 데 아주 강력합니다.
* **2. 주요 특징:** 여러 개의 상충하는 목적(휨 축소 vs 박리 축소)을 동시에 달성해야 할 때 전 세계 학계와 산업계에서 가장 널리 쓰이고 검증된 알고리즘입니다. 앞서 설명한 '파레토 비지배 정렬'을 내부적으로 탑재하여, 한 세대가 지날 때마다 더 우월한 유전자들만 선별해 살아남도록 설계되어 있습니다.
* **3. 작동 원리:** Step 4에서 뽑은 '초안' 설계들을 부모 유전자로 삼습니다. 이 부모들의 치수를 이리저리 섞고(교배), 가끔은 엉뚱하게 수치를 확 바꾸는(돌연변이) 과정을 거쳐 수백 개의 새로운 자식 설계도를 만듭니다. 그중 휨과 박리가 모두 우수한 엘리트 자식들만 살려내어 다시 교배를 시키는 과정을 100세대 이상 반복하여 완벽에 가까운 해를 찾아냅니다.
* **4. 프로젝트 도입 이유:** 딥러닝 역매핑(Step 4)이 단번에 던져준 초안은 방향성은 맞지만 소수점 단위의 정밀도는 떨어집니다. 따라서 이 초안의 주변 좁은 공간(±10%)을 알고리즘이 현미경처럼 샅샅이 뒤지며, 치수 제약조건에 한 치의 흠결도 없는 완벽한 최종 P1~P6 수치로 깎고 다듬기 위해 투입되었습니다.
* **5. 📺 참고 영상:** [유튜브: 유전 알고리즘(Genetic Algorithm) 어떻게 작동할까?](https://www.youtube.com/results?search_query=유전+알고리즘+쉽게+이해)

### 12. 강건 최적화 (Robust Optimization, $\mu + 2\sigma < Limit$)
* **1. 기법 소개:** 대리 모델(AI)이 틀릴 가능성, 즉 '오차 범위'까지 모두 최적화 계산식에 집어넣어, 공정상 변동이 생기거나 AI가 조금 틀리더라도 **최악의 상황에서 절대 파괴되지 않도록 뼈대를 튼튼하고 보수적으로 설계**하는 최고급 엔지니어링 철학입니다.
* **2. 주요 특징:** 평균적으로 성능이 좋은 설계를 찾는 것이 일반적인 최적화라면, 강건 최적화는 "재수 없게 모든 것이 엇나가는 최악의 경우에도 이 제품이 살아남을 수 있는가?"를 핵심 기준으로 삼습니다. 이를 통해 양산 단계에서의 수율(Yield) 저하를 막아냅니다.
* **3. 작동 원리:** 최적화 과정에서 GPR 모델이 "이 치수면 응력이 100이 나올 것 같습니다($\mu$)"라고 예측을 합니다. 하지만 여기서 멈추지 않고 "오차 범위가 $\pm 10$ 이니까($\sigma$), 최악의 경우를 상정해 응력이 120($\mu + 2\sigma$)까지 튈 수 있다고 치자. 이 120이라는 끔찍한 수치가 우리가 정한 파괴 한계선(Limit)을 넘는가?"를 가혹하게 검사합니다.
* **4. 프로젝트 도입 이유:** 만약 불확실성을 무시하고 AI의 단순 평균 예측값만 100% 맹신하여 한계선(Limit)에 아슬아슬하게 맞춘 설계를 뽑아낸다면, 나중에 실제 Ansys 시뮬레이션으로 검증할 때 보수적 마진이 없어 패키지가 터져버리는 대참사가 발생합니다. 이를 원천 차단하기 위해 이 수식을 안전장치로 걸었습니다.
* **5. 📺 참고 영상:** [유튜브: 강건 설계(Robust Design)와 식스시그마의 개념](https://www.youtube.com/results?search_query=강건+설계+최적화+개념)

### 13. Feasibility Rule (`pymoo` `G` Matrix)
* **1. 기법 소개:** 유전 알고리즘이 세대 교체를 하며 진화할 때, 규칙이나 제약 조건을 어긴 개체들을 무작정 실격 처리하여 버리는 대신, **"누가 얼마나 더 심하게 어겼는지"를 정교하게 비교 채점하여 올바른 진화의 방향(나침반)을 알려주는 벌점 처리 방식**입니다.
* **2. 주요 특징:** 기존의 무식한 최적화 방식은 제약조건을 하나라도 어기면 +999,999 같은 거대한 상수를 더해버립니다. 이 방식과 달리 Feasibility Rule은 위반의 크기를 연속적인 실수 값으로 추적하여, 비록 지금은 규칙을 어겼더라도 '조금 어긴 개체'가 '많이 어긴 개체'를 이기도록 시스템을 설계합니다.
* **3. 작동 원리:** 각 제약조건(Hard Constraints)을 평가하여, 규칙을 안전하게 만족하면 0 이하의 마이너스(-) 점수를 주고, 한계선을 초과하여 어기면 초과한 양에 비례하여 플러스(+) 점수를 매깁니다. 알고리즘은 이 점수(G 매트릭스)가 0 이하로 떨어지도록 유전자들을 이끌고 갑니다.
* **4. 프로젝트 도입 이유:** 위반 시 묻지마식 대형 벌점을 주면, 초기 세대의 모든 자식들이 벌점을 받게 되어 AI가 "대체 어느 방향으로 두께를 깎아야 살 수 있는지" 방향성(Gradient)을 잃고 멈춰버립니다. 살짝 어긴 개체를 서서히 안전한 구역으로 유도하여 부드럽게 수렴시키기 위해 도입했습니다.
* **5. 📺 참고 영상:** [유튜브: 파이썬 최적화 라이브러리 제약조건 처리법](https://www.youtube.com/results?search_query=pymoo+최적화+제약조건)

### 14. Knee Point (최적 밸런스 점 추출)
* **1. 기법 소개:** 수많은 1등 설계들이 모여 있는 파레토 프론티어 곡선(활 모양의 둥근 곡선) 중에서도, **가장 볼록하게 튀어나온 무릎(Knee) 위치에 해당하는 단 하나의 궁극적인 타협점을 콕 집어내는 최종 선택 기법**입니다. 설계자가 결정을 내리지 못할 때 수학이 내려주는 가장 합리적인 답안입니다.
* **2. 주요 특징:** 곡선의 양 끝단은 한쪽 성능에만 너무 치우쳐 있어서 하나를 얻기 위해 다른 하나를 막대하게 희생해야 합니다. 반면 무릎(Knee) 지점은 아주 약간의 희생만으로 두 가지 목표 모두에서 엄청난 이득을 취할 수 있는 '최고의 가성비 구간'입니다.
* **3. 작동 원리:** 휨(WarpMax) 점수와 박리(Peel) 점수의 스케일이 다르기 때문에, 이를 공평하게 0~1 사이의 값으로 평탄화(정규화)합니다. 그런 다음, 완벽한 이상향인 좌표 (0, 0) 원점으로부터 직선거리(유클리드 거리)를 측정하여, 거리가 가장 짧은 점을 수학적으로 찾아내어 랭크 1위로 지정합니다.
* **4. 프로젝트 도입 이유:** 휨만 극단적으로 줄이려고 덮개를 비정상적으로 깎거나, 반대로 박리만 줄이려고 극단적인 설계를 채택하는 편향된 의사결정을 방지하기 위함입니다. 두 가지 치명적 불량 요소를 가장 균형감 있게 방어하는 **'최고의 황금 밸런스 두께 조합 1개'**를 의사결정권자에게 자동으로 추천하기 위해 적용했습니다.
* **5. 📺 참고 영상:** [유튜브: 파레토 최적화에서 최적 해 선택 방법 (Knee Point)](https://www.youtube.com/results?search_query=파레토+프론티어+Knee+point)

# 차후 개선 방안


1. 2D 모델링 및 형상의 단순화, 재질 고정 등의 한계

본 프로젝트에서는 2D 모델링을 사용하여 기판 등 파트의 형상의 복잡성을 단순화 하였습니다.
또한, 팬에 의한 냉각 같은 유동 해석 문제도 간략화를 통해 생략했습니다.
실제 현실에 가까운 결과를 원할 경우 실제 3d 형상에 유동해석도 포함해야 합니다만
이 경우 실제 논문들에서 사용하는 슈퍼 컴퓨터를 사용하지 않으면 몇 개월을 24시간 풀로 돌려도 충분한 데이터를 얻을수 없습니다.

2. 패러미터 지정 

이번 실험에서 선정한 두께 패러미터는 실제로는 파운드리나 칩 설계사가 이미 고정해놓은 상수로 변경이 어렵습니다.

warpage의 경우 핵심 요인이 CTE(열팽창계수 차이), 온도변화량, 탄성계수, 두께 이기 때문에 EMC의 배합(CTE), 공정온도 (이번 실험에서는 120 -> 22도로 고정), si-bridge 두께나 substrtate의 내부 구조 등이 더 좋은 패러이터
(reference. Artificial Intelligence-Based Warpage Prediction Model for Accelerating Thermo-Mechanical Simulation in Advanced Packaging,  2025 IEEE 75th Electronic Components and Technology Conference (ECTC).)

두께의 경우 소숫점 단위의 최적 수치를 도출하더라도 절대 이와 정확한 수치를 적용하기 어렵습니다. 현실적으로 가공오차는 필연적이며 정밀가공을 하더라도 단가나 납기의 문제가 발생합니다.

3. 대리모델의 방향성

본래 기존의 대리모델의 개념에 맞게 구현하려면 모든 time step에 대응하는 시계열 데이터를 예측하고 구현하는 대리모델을 만들어야 하지만 이는 논문 수준의 매우 어려운 작업이기에 이번에는 절대값의 최대치만 예측하는 대리모델을 구현했습니다.

하지만 높은 정확도의 시계열 데이터를 예측하는 대리모델을 구현할 경우 step4 에서 학습시킬때 더 많은 데이터를 기반으로 더 정확한 결과를 도출할 수 있습니다.

다만 본 프로젝트와 같이 단순한 패러미터의 경우 과한 기능일수 있습니다.

4. 샘플링 방법 

본 프로젝트에서는 난수 생성 샘플링 기법을 Qusi-Random Search에 포함하는 Latin Hypercube Sampling(LHS)을 사용했습니다.
다음에는 adpative experimentation에 해당하는 bayesian optimization을 사용하여 더 뛰어난 효율성을 기대해볼수 있습니다.